# Bayesian Regression: Probabilistic Modeling and Uncertainty Quantification

This notebook explores Bayesian approaches to regression including uncertainty estimates and posterior distributions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score, mean_squared_error

# Generate data
np.random.seed(42)
X = np.sort(np.random.uniform(-5, 5, 100)).reshape(-1, 1)
y = np.sin(X).ravel() + np.random.randn(100) * 0.2

# Split data
X_train = X[:80]
y_train = y[:80]
X_test = X[80:]
y_test = y[80:]

# Bayesian Ridge with polynomial features
poly_features = PolynomialFeatures(degree=3)
X_train_poly = poly_features.fit_transform(X_train)
X_test_poly = poly_features.transform(X_test)

# Fit Bayesian Ridge
br = BayesianRidge(n_iter=1000, alpha_1=1e-6, alpha_2=1e-6, lambda_1=1e-6, lambda_2=1e-6)
br.fit(X_train_poly, y_train)

# Predictions with uncertainty
X_plot = np.linspace(-5, 5, 300).reshape(-1, 1)
X_plot_poly = poly_features.transform(X_plot)
y_mean, y_std = br.predict(X_plot_poly, return_std=True)

# Plot results
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_train, y_train, alpha=0.5, s=30, label='Training Data')
plt.scatter(X_test, y_test, alpha=0.5, s=30, label='Test Data', color='orange')
plt.plot(X_plot, y_mean, 'g-', linewidth=2, label='Mean Prediction')
plt.fill_between(X_plot.ravel(), 
                 y_mean - 2*y_std, 
                 y_mean + 2*y_std, 
                 alpha=0.3, color='green', label='95% Credible Interval')
plt.plot(X_plot, np.sin(X_plot), 'k--', linewidth=1, label='True Function')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Bayesian Ridge Regression with Uncertainty')
plt.legend()
plt.grid(True, alpha=0.3)

# Prediction uncertainty
y_test_mean, y_test_std = br.predict(X_test_poly, return_std=True)

plt.subplot(1, 2, 2)
plt.errorbar(X_test.ravel(), y_test_mean, yerr=2*y_test_std, fmt='o', alpha=0.6, label='Predictions with 95% CI')
plt.scatter(X_test, y_test, color='red', s=100, marker='x', linewidths=2, label='Actual Values')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Test Set Predictions with Uncertainty')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Model evaluation
train_r2 = br.score(X_train_poly, y_train)
test_r2 = br.score(X_test_poly, y_test)
test_mse = mean_squared_error(y_test, y_test_mean)

print(f"Train R²: {train_r2:.4f}")
print(f"Test R²: {test_r2:.4f}")
print(f"Test MSE: {test_mse:.4f}")
print(f"Estimated noise variance: {1/br.alpha_:.4f}")

# Bayesian Ridge Regression

Bayesian approach to Ridge regression with automatic relevance determination.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import BayesianRidge, ARDRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns

In [ ]:
# Generate dataset with noise
np.random.seed(101)
n_samples, n_features = 100, 12
X = np.random.randn(n_samples, n_features)

# Sparse true coefficients
true_coef = np.zeros(n_features)
true_coef[::3] = np.array([2.5, -1.8, 1.2, 0.9])

# Add heteroscedastic noise
noise_std = 0.1 + 0.05 * np.abs(X[:, 0])
y = X @ true_coef + noise_std * np.random.randn(n_samples)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Bayesian Ridge
br = BayesianRidge()
br.fit(X_train, y_train)
y_pred_br = br.predict(X_test)

# ARD Regression
ard = ARDRegression()
ard.fit(X_train, y_train)
y_pred_ard = ard.predict(X_test)

print(f"Bayesian Ridge MSE: {mean_squared_error(y_test, y_pred_br):.4f}")
print(f"ARD MSE: {mean_squared_error(y_test, y_pred_ard):.4f}")
print(f"ARD selected features: {np.sum(np.abs(ard.coef_) > 0.1)}")